# 20. 오늘 제출 전 점검: 제출 파일의 섞는 비율 + LR_trunc의 test 동작

## 배경
- 11/11 마지막 제출로 16번 6개 버전(핫스팟 제외) 파일을 낼 예정
  - 비율: LR_trunc 0.38 · RF 0.31 · XGB 0.23 · ET 0.08 → LR_trunc 비중이 가장 큼
- 확인할 것 2가지
  1. 실제 제출 파일(16번 6개·7개 버전)이 어떤 비율로 만들어졌는지 정확한 값(Caruana 선택 횟수)을 기록
     - 16번은 실행 환경에 따라 다른 조합이 골라짐 (16 결과의 처음 실행 0.38/0.31/0.23/0.08 vs 18 PR 재실행 0.36/0.29/0.21/0.14)
     - 재현 스크립트(PR #26)에 16 프리셋을 추가할 때, **제출 파일을 그대로 다시 만드는 비율**이 무엇인지 확정하기 위함
  2. trunc 피처가 정규화 후 test에서도 train과 같은 방식으로 계산되는지 (정규화 전 07 F는 test의 `X` 표기를 못 읽어 LB 효과가 없었음)

## 방법
- 1: 제출 파일을 만든 16번 캐시(`submissions/cache16/`, 처음 실행 환경)의 OOF로 Caruana 선택을 다시 실행해 **선택 횟수**를 확인 → 그 비율로 섞은 test 예측이 제출 파일과 같은지 비교
- 2: train·test 각각에서 변이 토큰 유형 비율, 절단 변이 비율, 분류 못 한 토큰을 집계 + LR / LR_trunc 예측이 바뀌는 정도를 CV(OOF)와 test에서 비교
- **test는 학습·튜닝에 쓰지 않음**: 피처가 제대로 계산되는지 확인하는 진단용 집계만 하고, 결과로 모델·피처·비율을 바꾸지 않음


In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd
from src.config import WT
from src.data import load_data, get_gene_cols
from src.features import mutation_type, mutation_features
from src.validation import encode_target, get_folds, macro_f1

train, test, submission = load_data()          # 16과 동일 (정규화 + 이소폼 중복 제거)
genes = get_gene_cols(train)
y, le = encode_target(train)
FOLDS = get_folds(y)
CACHE = Path('../submissions/cache16')
NAMES = ['XGB', 'XGB_hs', 'XGB_nostats', 'LR', 'LR_trunc', 'RF', 'ET']
P_oof  = {k: np.load(CACHE / f'{k}.npz')['oof']  for k in NAMES}
P_test = {k: np.load(CACHE / f'{k}.npz')['test'] for k in NAMES}
print('캐시 로드:', list(P_oof))

캐시 로드: ['XGB', 'XGB_hs', 'XGB_nostats', 'LR', 'LR_trunc', 'RF', 'ET']


## 1) 제출 파일의 섞는 비율 확인
- 16번과 같은 `caruana()`를 OOF 전체에 실행하고, 비율 대신 **선택 횟수**를 출력
- 기록 비율(PR #26 첫 버전 프리셋과 같은 값): 6개 = LR_trunc 5, RF 4, XGB 3, ET 1 (÷13) / 7개 = XGB_hs 7, LR_trunc 7, RF 5, ET 1 (÷20)

In [2]:
mix = lambda w, P: sum(v * P[k] for k, v in w.items())

def caruana_counts(pool, idx, n_iter=20):
    """16번 caruana()와 같은 규칙. 반환: 선택 횟수 Counter"""
    score = lambda c: macro_f1(y[idx], mix({k: v / sum(c.values()) for k, v in c.items()},
                                            {k: P_oof[k][idx] for k in c}).argmax(1))
    first = max(pool, key=lambda k: score({k: 1}))
    counts, best_counts, best = Counter({first: 1}), Counter({first: 1}), score({first: 1})
    for _ in range(n_iter - 1):
        cand = max(pool, key=lambda k: score(counts + Counter({k: 1})))
        counts = counts + Counter({cand: 1})
        s = score(counts)
        if s > best:
            best, best_counts = s, Counter(counts)
    return best_counts

RECORD = {  # 제출 파일 비율 (Caruana 선택 횟수)
    'select6': {'LR_trunc': 5, 'RF': 4, 'XGB': 3, 'ET': 1},
    'select7': {'XGB_hs': 7, 'LR_trunc': 7, 'RF': 5, 'ET': 1}}
POOLS = {'select6': [k for k in NAMES if k != 'XGB_hs'], 'select7': NAMES}
FILES = {'select6': '1111_kangkiwi_ensemble_select6_nohs_0.427.csv', 'select7': '1111_kangkiwi_ensemble_select7_0.432.csv'}
idx = np.arange(len(y))
for name, pool in POOLS.items():
    cnt = dict(caruana_counts(pool, idx))
    same = cnt == RECORD[name]
    print(f'[{name}] 노트북 선택 횟수 {cnt} (합 {sum(cnt.values())}) | 기록 비율과 {"일치" if same else "다름"}')
    w = {k: v / sum(RECORD[name].values()) for k, v in RECORD[name].items()}
    f = Path('../submissions') / FILES[name]
    if f.exists():
        agree = (le.inverse_transform(mix(w, P_test).argmax(1)) == pd.read_csv(f).SUBCLASS.values).mean()
        print(f'         기록 비율로 섞은 test 예측 vs 제출 파일 {f.name}: {agree:.1%} 일치')

[select6] 노트북 선택 횟수 {'XGB': 3, 'LR_trunc': 5, 'ET': 1, 'RF': 4} (합 13) | 기록 비율과 일치
         기록 비율로 섞은 test 예측 vs 제출 파일 1111_kangkiwi_ensemble_select6_nohs_0.427.csv: 100.0% 일치


[select7] 노트북 선택 횟수 {'XGB_hs': 7, 'LR_trunc': 7, 'RF': 5, 'ET': 1} (합 20) | 기록 비율과 일치
         기록 비율로 섞은 test 예측 vs 제출 파일 1111_kangkiwi_ensemble_select7_0.432.csv: 100.0% 일치


## 2) trunc 피처가 test에서 제대로 계산되는지
- 변이 토큰 유형 비율, 변이가 있는 칸 중 절단 변이(nonsense·frameshift) 칸 비율, 분류 못 한 토큰(`other`)

In [3]:
def type_summary(df):
    c, other, cells, tcells = Counter(), Counter(), 0, 0
    for row in df[genes].to_numpy():
        for v in row:
            if v == WT:
                continue
            cells += 1
            types = [mutation_type(t) for t in v.split()]
            c.update(types)
            other.update(t for t, ty in zip(v.split(), types) if ty == 'other')
            tcells += any(ty in ('nonsense', 'frameshift') for ty in types)
    tot = sum(c.values())
    return {**{k: round(100 * v / tot, 2) for k, v in c.most_common()},
            '절단 칸 %': round(100 * tcells / cells, 2), '미분류 토큰 수': sum(other.values())}

pd.DataFrame({'train': type_summary(train), 'test': type_summary(test)}).fillna(0)

,train,test
missense,64.70,60.92
silent,26.10,26.90
nonsense,5.26,3.70
frameshift,3.95,6.69
inframe_indel,0.00,1.72
절단 칸 %,10.17,10.49
미분류 토큰 수,0.00,1.00
stop_loss,0.00,0.07
other,0.00,0.00


In [4]:
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
T_tr, T_te = F_tr['trunc'], F_te['trunc']
unseen = (T_tr.sum(axis=0) == 0) & (T_te.sum(axis=0) > 0)
print(f'trunc / any 비율: train {T_tr.values.sum() / F_tr["any"].values.sum():.4f} | test {T_te.values.sum() / F_te["any"].values.sum():.4f}')
print(f'train에 없고 test에만 1이 있는 trunc 열: {int(unseen.sum())}개 (test 절단 변이의 {T_te.loc[:, unseen].values.sum() / T_te.values.sum():.1%}) → LR 가중치 0이라 영향 없음')

d = pd.DataFrame({'train %': T_tr.mean() * 100, 'test %': T_te.mean() * 100})
d['차이'] = d['test %'] - d['train %']
d.sort_values('차이', ascending=False).head(8).round(1)
print(f"train에서 APC 절단 변이 환자 중 COAD 비율: {train.SUBCLASS[T_tr['APC__trunc'] == 1].eq('COAD').mean():.0%}")
print(f"대략적인 COAD 비율 (test APC 절단 변이 보유율 × train에서 그중 COAD 비율): {T_te['APC__trunc'].mean() * train.SUBCLASS[T_tr['APC__trunc'] == 1].eq('COAD').mean():.1%}  ← 진단용, 예측 조정에 쓰지 않음")

trunc / any 비율: train 0.1017 | test 0.1049
train에 없고 test에만 1이 있는 trunc 열: 417개 (test 절단 변이의 5.6%) → LR 가중치 0이라 영향 없음
train에서 APC 절단 변이 환자 중 COAD 비율: 71%
대략적인 COAD 비율 (test APC 절단 변이 보유율 × train에서 그중 COAD 비율): 17.2%  ← 진단용, 예측 조정에 쓰지 않음


In [5]:
dist = lambda p: pd.Series(le.inverse_transform(p.argmax(1))).value_counts(normalize=True)
W6 = {'LR_trunc': 5 / 13, 'RF': 4 / 13, 'XGB': 3 / 13, 'ET': 1 / 13}
print(f"LR vs LR_trunc 예측이 다른 비율: OOF {(P_oof['LR'].argmax(1) != P_oof['LR_trunc'].argmax(1)).mean():.1%} | "
      f"test {(P_test['LR'].argmax(1) != P_test['LR_trunc'].argmax(1)).mean():.1%}")
tab = pd.DataFrame({'train': train.SUBCLASS.value_counts(normalize=True), 'LR': dist(P_test['LR']),
                    'LR_trunc': dist(P_test['LR_trunc']), 'XGB': dist(P_test['XGB']), 'select6': dist(mix(W6, P_test))})
tab.fillna(0).mul(100).round(1).sort_values('select6', ascending=False).head(8)

LR vs LR_trunc 예측이 다른 비율: OOF 8.4% | test 8.8%


,train,LR,LR_trunc,XGB,select6
COAD,3.6,22.5,22.8,11.6,22.5
STES,6.1,7.9,8.1,13.1,10.8
OV,4.1,6.5,6.8,4.9,6.0
KIPAN,8.3,3.1,3.5,3.8,5.9
BRCA,12.7,6.2,6.0,4.7,5.3
TGCT,2.0,3.6,3.4,5.2,3.8
LGG,3.7,3.5,3.6,3.5,3.8
PCPG,2.4,4.5,4.0,3.7,3.7


## 결과
**1) 비율**
- 제출 파일을 만든 캐시 기준 선택 횟수
  - select6: LR_trunc 5, RF 4, XGB 3, ET 1 (÷13) / select7: XGB_hs 7, LR_trunc 7, RF 5, ET 1 (÷20)
- 이 비율로 섞은 test 예측이 선영 컴퓨터에서 만든 파일 2개와 **100% 일치**
- ⚠️ **실제 LB 제출 파일은 설아님 데스크탑에서 16번을 돌려 만든 파일**: RF 5, LR_trunc 4, XGB 3, ET 2 (÷14). 위 표의 "100% 일치"는 선영 컴퓨터 비율(÷13) 기준
- ⚠️ 이 비율은 처음 실행(16 PR) 캐시 기준. 18 PR 재실행 환경에서는 RF 5, LR_trunc 4, XGB 3, ET 2 (÷14)로 다른 조합이 골라짐 (XGB 확률 차이)
  - 재현 스크립트에 16 프리셋을 넣을 때는 재실행 비율이 아니라 **위 분수(÷13, ÷20)**를 넣어야 제출 파일과 같은 비율이 됨
  - 단, XGB 확률은 실행 환경(CPU)에 따라 조금 달라서, 다른 PC에서 만든 파일은 예측이 아주 약간 다를 수 있음

**2) LR_trunc의 test 동작**

| 확인 | train | test |
| :-- | :-: | :-: |
| 변이 칸 중 절단 변이 칸 | 10.2% | 10.5% |
| 분류 못 한 토큰 | 0개 | 1개 (`M391`) |
| LR과 LR_trunc 예측이 다른 비율 | 8.4% (OOF) | 8.8% |

- 정규화 후 `X` 종결 코돈과 `fs` 표기를 모두 절단 변이로 인식 → trunc가 test에서도 train과 같은 방식으로 계산됨
- trunc를 넣었을 때 예측이 바뀌는 정도가 CV와 test에서 비슷 → test에서만 이상하게 움직이지 않음
- train에 없던 trunc 열(417개, test 절단 변이의 5.6%)은 LR 가중치가 0이라 영향 없음

**참고: test에서 크게 늘어난 절단 변이**
- APC(3.5% → 24.2%), SOX9, TCF7L2, ACVR2A, FBXW7: 모두 대장암 대표 유전자
- 비율은 "환자 중 그 유전자에 절단 변이가 있는 환자 비율" (예: test 환자의 24.2%가 APC 절단 변이 보유)
- train에서 APC 절단 변이 환자의 71%가 COAD → 그대로 곱하면 24.2% × 0.71 ≈ **17%**
  - LB가 가장 좋았던 14 섞기의 COAD 예측 17.3%와 가깝고, select6의 22.5%는 이보다 많음 → **select6이 COAD를 너무 많이 예측했다는 근거**
  - APC 없는 COAD, APC가 있는 다른 암종도 있어서 대략적인 계산 (진단용, 예측 조정에는 쓰지 않음)
- trunc 없는 LR도 COAD 22.5% → COAD 비율은 LR_trunc 때문이 아님

## 결론
- 제출 파일 비율: 선영 컴퓨터 ÷13 (LR_trunc 5, RF 4, XGB 3, ET 1) / 실제 LB 제출은 설아님 데스크탑 ÷14 (RF 5, LR_trunc 4, XGB 3, ET 2)
- **select6 LB 0.3183**: 14 섞기 0.3247보다 −0.006, 10 + 정규화 0.3202보다도 낮음 → 최종 후보 아님, 14 유지
- trunc는 test에서도 정상 계산됐으므로 계산 오류가 아니라, LR_trunc 비중이 큰 조합의 CV 우위가 LB로 이어지지 않은 것
- 오늘 제출을 COAD 예측 비율로 보면 12.4% (10 + 정규화, 0.3202) → 17.3% (14, 0.3247) → 22.5% (select6, 0.3183). APC 집계로 본 COAD 대략치(약 17%)와 가까운 14가 가장 높음
- LR_trunc는 정규화 후 test에서 정상 작동 (피처 계산 기준). 하지만 LR_trunc 비중이 큰 select6은 LB에서 낮았음
- 이 노트북의 test 집계는 진단용이며, 결과로 모델·피처·비율을 바꾸지 않음
